# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imnxr/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions + reason codes

This playbook ranks content already observed as declining in the current measurement window. The action score combines three percentile-ranked signals: decline strength (50%), engagement rate (30%), and days since the last update (20%).

The queue is decision-support for manual review, not an automatic refresh list. Higher-ranked items show a stronger observed decline and may also have higher engagement or be older since their last update.

Reason codes explain why an item appears in the queue:

- `SEVERE_DECLINE`, `MODERATE_DECLINE`, or `MILD_DECLINE` describe the measured current-window decline.
- `HIGH_ENGAGEMENT` indicates engagement at or above the declining-candidate 75th percentile.
- `STALE_CONTENT` indicates days since last update at or above the declining-candidate 75th percentile.

These codes describe observed dataset conditions. They do not prove that refreshing a page will improve future performance.

In [19]:
from pathlib import Path

import numpy as np
import pandas as pd

# Make paths work whether VS Code starts from the repository root
# or from the work/notebooks folder.
repo_root = Path.cwd()

if repo_root.name == "notebooks":
    repo_root = repo_root.parent.parent

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
output_dir = repo_root / "work" / "outputs"

print("Repository root:", repo_root)
print("Dataset exists:", data_path.exists())
print("Output directory:", output_dir)

Repository root: d:\Internship\flyrank-ml-internship
Dataset exists: True
Output directory: d:\Internship\flyrank-ml-internship\work\outputs


In [20]:
df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("Number of columns:", len(df.columns))
print("\nFirst 10 column names:")
print(df.columns[:10].tolist())

Dataset shape: (30000, 44)
Number of columns: 44

First 10 column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count']


In [21]:
important_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "engagement_rate",
    "word_count",
    "content_age_days",
    "days_since_update",
]

for column in important_columns:
    print(f"{column}: {column in df.columns}")
    

content_id: True
client_id: True
trend_direction: True
trend_pct: True
engagement_rate: True
word_count: True
content_age_days: True
days_since_update: False


In [22]:
time_columns = [
    column
    for column in df.columns
    if any(word in column.lower() for word in ["age", "day", "date", "update"])
]

print("Possible time-related columns:")
print(time_columns)

Possible time-related columns:
['pageviews_90d', 'engaged_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'engagement_rate']


In [23]:
queue_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
]

column_check = pd.DataFrame({
    "dtype": df[queue_columns].dtypes.astype(str),
    "missing": df[queue_columns].isna().sum(),
    "unique_values": df[queue_columns].nunique(),
})

column_check


,dtype,missing,unique_values
content_id,str,0,30000
client_id,str,0,32
trend_direction,str,0,5
trend_pct,float64,3388,2712
engagement_rate,float64,0,915
content_age_days,int64,0,225
days_since_last_update,int64,0,57


In [24]:
print("Trend direction counts:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nMissing trend_pct by trend direction:")
print(
    df.groupby("trend_direction")["trend_pct"]
      .apply(lambda series: series.isna().sum())
      .sort_values(ascending=False)
)

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Missing trend_pct by trend direction:
trend_direction
new       2236
flat      1152
down         0
stable       0
up           0
Name: trend_pct, dtype: int64


In [25]:
df["decline_strength"] = np.where(
    df["trend_direction"].eq("down"),
    df["trend_pct"].abs(),
    0.0,
)

print(df["decline_strength"].describe())
print("\nRows with positive decline strength:", (df["decline_strength"] > 0).sum())

count    30000.000000
mean        31.501570
std         33.725563
min          0.000000
25%          0.000000
50%         26.000000
75%         58.700000
max        100.000000
Name: decline_strength, dtype: float64

Rows with positive decline strength: 16262


In [26]:
ranking_inputs = [
    "decline_strength",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
]

df[ranking_inputs].describe().T

,count,mean,std,min,25%,50%,75%,max
decline_strength,30000.0,31.50157,33.725563,0.0,0.0,26.0,58.70,100.0
engagement_rate,30000.0,2.53452,8.310096,0.0,0.0,0.0,1.35,100.0
content_age_days,30000.0,256.16780,132.707930,90.0,132.0,236.0,333.00,564.0
days_since_last_update,30000.0,46.09830,42.078709,1.0,20.0,20.0,104.00,373.0


In [27]:
candidates = df.loc[df["trend_direction"].eq("down")].copy()

candidates["decline_rank"] = candidates["decline_strength"].rank(pct=True)
candidates["engagement_rank"] = candidates["engagement_rate"].rank(pct=True)
candidates["staleness_rank"] = candidates["days_since_last_update"].rank(pct=True)

print("Review candidates:", len(candidates))

candidates[
    ["decline_rank", "engagement_rank", "staleness_rank"]
].describe().T

Review candidates: 16262


,count,mean,std,min,25%,50%,75%,max
decline_rank,16262.0,0.500031,0.288592,0.000154,0.250092,0.499385,0.750154,0.957139
engagement_rank,16262.0,0.500031,0.227430,0.361979,0.361979,0.361979,0.750246,0.998739
staleness_rank,16262.0,0.500031,0.274514,0.002275,0.331970,0.331970,0.823269,0.999969


In [28]:
candidates["action_score"] = (
    0.50 * candidates["decline_rank"]
    + 0.30 * candidates["engagement_rank"]
    + 0.20 * candidates["staleness_rank"]
)

print(candidates["action_score"].describe())

count    16262.000000
mean         0.500031
std          0.151753
min          0.112240
25%          0.386056
50%          0.507865
75%          0.608142
max          0.967021
Name: action_score, dtype: float64


In [29]:
candidates = candidates.sort_values(
    "action_score",
    ascending=False,
).reset_index(drop=True)

candidates["priority_rank"] = np.arange(1, len(candidates) + 1)

top_columns = [
    "priority_rank",
    "content_id",
    "client_id",
    "action_score",
    "decline_strength",
    "engagement_rate",
    "days_since_last_update",
]

candidates[top_columns].head(10)

,priority_rank,content_id,client_id,action_score,decline_strength,engagement_rate,days_since_last_update
0,1,content_d6d5bc71c047,client_9f14025af0,0.967021,100.0,16.67,151
1,2,content_6fffca7b4d3d,client_4ec9599fc2,0.942845,100.0,100.00,104
2,3,content_83dba2842fe6,client_8527a891e2,0.941563,100.0,50.00,104
3,4,content_9576e5dcef33,client_3fdba35f04,0.939552,100.0,33.33,104
4,5,content_973754ecfb28,client_02d20bbd7e,0.938473,100.0,28.57,104
5,6,content_33c54dccd63c,client_624b60c58c,0.937385,100.0,25.00,104
6,7,content_696352d1b5c5,client_8527a891e2,0.934996,100.0,20.00,104
7,8,content_39cda9327033,client_8722616204,0.931435,100.0,15.15,104
8,9,content_45c48b701376,client_8527a891e2,0.925716,100.0,11.11,104
9,10,content_331a01416e92,client_4ec9599fc2,0.925716,100.0,11.11,104


In [30]:
def build_reason_codes(row):
    reasons = []

    if row["decline_strength"] >= 75:
        reasons.append("SEVERE_DECLINE")
    elif row["decline_strength"] >= 40:
        reasons.append("MODERATE_DECLINE")
    else:
        reasons.append("MILD_DECLINE")

    if row["engagement_rate"] >= candidates["engagement_rate"].quantile(0.75):
        reasons.append("HIGH_ENGAGEMENT")

    if row["days_since_last_update"] >= candidates["days_since_last_update"].quantile(0.75):
        reasons.append("STALE_CONTENT")

    return " | ".join(reasons)


candidates["reason_codes"] = candidates.apply(build_reason_codes, axis=1)

candidates[
    [
        "priority_rank",
        "content_id",
        "action_score",
        "decline_strength",
        "engagement_rate",
        "days_since_last_update",
        "reason_codes",
    ]
].head(10)

,priority_rank,content_id,action_score,decline_strength,engagement_rate,days_since_last_update,reason_codes
0,1,content_d6d5bc71c047,0.967021,100.0,16.67,151,SEVERE_DECLINE | HIGH_ENGAGEMENT | STALE_CONTENT
1,2,content_6fffca7b4d3d,0.942845,100.0,100.00,104,SEVERE_DECLINE | HIGH_ENGAGEMENT | STALE_CONTENT
2,3,content_83dba2842fe6,0.941563,100.0,50.00,104,SEVERE_DECLINE | HIGH_ENGAGEMENT | STALE_CONTENT
3,4,content_9576e5dcef33,0.939552,100.0,33.33,104,SEVERE_DECLINE | HIGH_ENGAGEMENT | STALE_CONTENT
4,5,content_973754ecfb28,0.938473,100.0,28.57,104,SEVERE_DECLINE | HIGH_ENGAGEMENT | STALE_CONTENT
5,6,content_33c54dccd63c,0.937385,100.0,25.00,104,SEVERE_DECLINE | HIGH_ENGAGEMENT | STALE_CONTENT
6,7,content_696352d1b5c5,0.934996,100.0,20.00,104,SEVERE_DECLINE | HIGH_ENGAGEMENT | STALE_CONTENT
7,8,content_39cda9327033,0.931435,100.0,15.15,104,SEVERE_DECLINE | HIGH_ENGAGEMENT | STALE_CONTENT
8,9,content_45c48b701376,0.925716,100.0,11.11,104,SEVERE_DECLINE | HIGH_ENGAGEMENT | STALE_CONTENT
9,10,content_331a01416e92,0.925716,100.0,11.11,104,SEVERE_DECLINE | HIGH_ENGAGEMENT | STALE_CONTENT


In [31]:
reason_summary = (
    candidates["reason_codes"]
    .value_counts()
    .rename_axis("reason_codes")
    .reset_index(name="count")
)

reason_summary

,reason_codes,count
0,MODERATE_DECLINE,3595
1,SEVERE_DECLINE,2757
2,MODERATE_DECLINE | STALE_CONTENT,1995
3,MILD_DECLINE,1875
4,MODERATE_DECLINE | HIGH_ENGAGEMENT,1245
5,MILD_DECLINE | STALE_CONTENT,1008
6,SEVERE_DECLINE | STALE_CONTENT,966
7,MILD_DECLINE | HIGH_ENGAGEMENT,836
8,MODERATE_DECLINE | HIGH_ENGAGEMENT | STALE_CON...,761
9,MILD_DECLINE | HIGH_ENGAGEMENT | STALE_CONTENT,676


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and limits

The intended users are content strategists, SEO analysts, and editors who need a prioritized list of pages for manual review.

The queue may be used to:

- decide which declining pages to inspect first;
- support editorial planning;
- compare observed decline, engagement, and staleness;
- document why an item was prioritized.

The queue is valid only for the dataset, fields, and measurement window used here. It identifies observed current-window decline, not future decline.

It should not be used to:

- automatically refresh, delete, redirect, or publish content;
- claim that a refresh will improve traffic or rankings;
- evaluate clients outside the represented data without new validation;
- replace checks of search intent, content quality, business value, seasonality, and technical SEO.

The score is a prioritization rule, not a probability and not a causal estimate.

In [32]:
intended_use_check = pd.Series({
    "queue_contains_only_declining_items": candidates["trend_direction"].eq("down").all(),
    "action_score_is_probability": False,
    "automatic_action_allowed": False,
    "human_review_required": True,
})

intended_use_check

queue_contains_only_declining_items     True
action_score_is_probability            False
automatic_action_allowed               False
human_review_required                   True
dtype: bool

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review + the no-go list

Before acting on a recommendation, a reviewer should inspect the page, query context, recent changes, traffic quality, search intent, seasonality, business importance, and technical issues. The reviewer should also confirm that the decline is not explained by tracking errors, migrations, campaign changes, or known reporting gaps.

A person must decide whether the right action is to refresh, consolidate, redirect, leave unchanged, investigate further, or remove the item from the queue.

The system must never automatically:

- publish or rewrite content;
- delete, redirect, or deindex pages;
- change titles, metadata, internal links, or canonical tags;
- contact clients or assign blame;
- treat the score as proof of poor quality;
- claim that a refresh will improve rankings, traffic, revenue, or engagement;
- prioritize pages containing sensitive, legal, medical, financial, or reputation-critical information without specialist review.

Low-confidence, unusual, or high-impact cases should always be escalated to a human reviewer.

In [33]:
no_go_check = pd.Series({
    "auto_publish": False,
    "auto_delete_or_redirect": False,
    "auto_change_metadata": False,
    "auto_contact_clients": False,
    "claim_guaranteed_improvement": False,
    "specialist_review_for_sensitive_content": True,
})

no_go_check

auto_publish                               False
auto_delete_or_redirect                    False
auto_change_metadata                       False
auto_contact_clients                       False
claim_guaranteed_improvement               False
specialist_review_for_sensitive_content     True
dtype: bool

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring / retrain triggers

The recommendations may become stale when the data distribution, client mix, content strategy, tracking setup, or search environment changes.

The queue should be reviewed or rebuilt when:

- the share of declining pages changes substantially;
- action-score distributions shift noticeably;
- engagement-rate or staleness distributions move outside their previous ranges;
- new clients or content types appear;
- tracking definitions or source systems change;
- editors report that top-ranked items are repeatedly not useful;
- observed performance on later data is materially worse than the grouped validation results;
- enough new labeled data is available to support a fresh grouped or time-based evaluation.

A retrain or redesign should not be triggered only because one page performs unexpectedly. The decision should use repeated evidence across batches, clients, or time periods.

In [36]:
monitoring_summary = pd.Series({
    "declining_share": df["trend_direction"].eq("down").mean(),
    "median_action_score": candidates["action_score"].median(),
    "top_decile_cutoff": candidates["action_score"].quantile(0.90),
    "median_engagement_rate": candidates["engagement_rate"].median(),
    "median_days_since_last_update": candidates["days_since_last_update"].median(),
    "number_of_clients": df["client_id"].nunique(),
    "number_of_candidates": len(candidates),
})

monitoring_summary

declining_share                      0.542067
median_action_score                  0.507865
top_decile_cutoff                    0.694314
median_engagement_rate               0.000000
median_days_since_last_update       20.000000
number_of_clients                   32.000000
number_of_candidates             16262.000000
dtype: float64

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for the paper

The ranked action queue is exported to:

`work/outputs/w07_content_action_queue.csv`

The file contains 16,262 observed declining-content candidates. Each row includes its priority rank, anonymized content and client identifiers, action score, observed trend values, engagement, content age, time since last update, and human-readable reason codes.

The exported score is a relative prioritization score within this dataset. It is not a probability of future decline or guaranteed improvement after a refresh. The file is intended to support the final research webpage, summary tables, and manual review examples.

In [34]:
export_columns = [
    "priority_rank",
    "content_id",
    "client_id",
    "action_score",
    "trend_direction",
    "trend_pct",
    "decline_strength",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
    "reason_codes",
]

action_queue = candidates[export_columns].copy()

print("Export rows:", len(action_queue))
print("Export columns:", len(action_queue.columns))

action_queue.head()

Export rows: 16262
Export columns: 11


,priority_rank,content_id,client_id,action_score,trend_direction,trend_pct,decline_strength,engagement_rate,content_age_days,days_since_last_update,reason_codes
0,1,content_d6d5bc71c047,client_9f14025af0,0.967021,down,-100.0,100.0,16.67,335,151,SEVERE_DECLINE | HIGH_ENGAGEMENT | STALE_CONTENT
1,2,content_6fffca7b4d3d,client_4ec9599fc2,0.942845,down,-100.0,100.0,100.00,445,104,SEVERE_DECLINE | HIGH_ENGAGEMENT | STALE_CONTENT
2,3,content_83dba2842fe6,client_8527a891e2,0.941563,down,-100.0,100.0,50.00,275,104,SEVERE_DECLINE | HIGH_ENGAGEMENT | STALE_CONTENT
3,4,content_9576e5dcef33,client_3fdba35f04,0.939552,down,-100.0,100.0,33.33,263,104,SEVERE_DECLINE | HIGH_ENGAGEMENT | STALE_CONTENT
4,5,content_973754ecfb28,client_02d20bbd7e,0.938473,down,-100.0,100.0,28.57,332,104,SEVERE_DECLINE | HIGH_ENGAGEMENT | STALE_CONTENT


In [35]:
output_dir.mkdir(parents=True, exist_ok=True)

queue_path = output_dir / "w07_content_action_queue.csv"
action_queue.to_csv(queue_path, index=False)

print("Saved:", queue_path)
print("File exists:", queue_path.exists())
print("Rows written:", len(action_queue))

Saved: d:\Internship\flyrank-ml-internship\work\outputs\w07_content_action_queue.csv
File exists: True
Rows written: 16262


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.